# Домашнее задание 1. Бандиты и уравнения Беллмана

**Вес в оценке:** базовое ДЗ, среднее по сложности.

Задание состоит из двух частей:

1. **Практика (70%)** — реализовать и сравнить бандит-алгоритмы
2. **Теория (30%)** — вывести уравнения Беллмана и посчитать return для простых MDP

Части задания с `assert` проверяются автоматически при запуске ячейки — если assert не
упал, эта часть засчитана. Часть заданий (теория, графики) проверяется вручную.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)


## Часть 1. Практика: бандит-алгоритмы (70 баллов)

### 1.1 ε-greedy с убывающим ε (15 баллов)

На семинаре мы реализовали ε-greedy с постоянным ε. Проблема: даже когда агент уверен
в лучшей руке, он продолжает исследовать с той же частотой ε, из-за чего regret растёт
линейно, а не логарифмически.

**Задание:** реализуйте `DecayingEpsilonGreedyAgent`, где ε убывает со временем,
например ε_t = min(1, c / t) для некоторой константы c, или экспоненциальное затухание.
Класс должен иметь тот же интерфейс, что и агенты с семинара: `select_arm()`, `update(arm, reward)`.


In [ ]:
class DecayingEpsilonGreedyAgent:
    def __init__(self, n_arms, c=1.0, rng=None):
        self.n_arms = n_arms
        self.c = c
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.t = 0
        self.rng = rng or np.random.default_rng()

    def _epsilon(self) -> float:
        # TODO: реализуйте расписание epsilon_t, например min(1, c / (t+1))
        raise NotImplementedError

    def select_arm(self) -> int:
        # TODO
        raise NotImplementedError

    def update(self, arm: int, reward: float):
        # TODO: такое же инкрементальное обновление Q, как у EpsilonGreedyAgent на семинаре
        raise NotImplementedError


In [ ]:
# Самопроверка: агент должен постепенно приближаться к чисто жадному поведению
_agent = DecayingEpsilonGreedyAgent(n_arms=3, c=5.0, rng=np.random.default_rng(1))
for _ in range(1000):
    a = _agent.select_arm()
    assert 0 <= a < 3
    _agent.update(a, reward=float(a == 1))

assert _agent._epsilon() < 0.1, "после 1000 шагов epsilon должен стать маленьким"
print("OK: DecayingEpsilonGreedyAgent проходит базовую проверку")


### 1.2 Сравнение на нескольких конфигурациях (25 баллов)

На семинаре мы сравнивали агентов на одной конфигурации рук. Постройте сравнение
(средний накопленный regret) для **трёх** разных конфигураций `bandit_probs`:

1. "Лёгкая" — руки сильно отличаются, например `[0.1, 0.9]`
2. "Сложная" — руки почти одинаковые, например `[0.45, 0.5, 0.48]`
3. "Много рук" — например 10 рук со случайными вероятностями

Сравните **4 агентов**: `EpsilonGreedyAgent` (константный ε), `DecayingEpsilonGreedyAgent`,
`UCB1Agent`, `ThompsonSamplingAgent` (используйте реализации с семинара — скопируйте их сюда
или импортируйте, если оформили семинар как модуль).

Постройте 3 графика (по одному на конфигурацию), на каждом — 4 кривые regret.


In [ ]:
# Скопируйте сюда классы EpsilonGreedyAgent, UCB1Agent, ThompsonSamplingAgent
# и функцию run_agent из семинара, либо импортируйте их.

# TODO: ваш код здесь


**Вопрос (ответьте текстом в этой ячейке):** на какой из трёх конфигураций разница между
UCB1/Thompson и ε-greedy наиболее заметна, и почему? Что происходит с "лёгкой"
конфигурацией при увеличении числа рук с явно плохими вероятностями?

_Ваш ответ:_ TODO


### 1.3 Устойчивость к гиперпараметрам (10 баллов)

Постройте график зависимости **итогового** (на последнем шаге) среднего regret от
гиперпараметра:

* для ε-greedy — от константного ε ∈ {0.01, 0.05, 0.1, 0.2, 0.5}
* для UCB1 — от c ∈ {0.5, 1, 2, 4, 8}

Сделайте вывод о том, насколько результат чувствителен к выбору гиперпараметра
у каждого метода.


In [ ]:
# TODO: ваш код здесь


### 1.4 (бонус, 20 баллов) Non-stationary бандит

Что если вероятности рук **меняются со временем**? Реализуйте среду, где `probs` дрейфуют
(например, раз в 500 шагов случайно перемешиваются, или на каждом шаге добавляется малый
гауссовский шум с последующим клипом в [0, 1]).

Сравните на такой среде обычное усреднение Q (как в `EpsilonGreedyAgent`) с
**экспоненциально взвешенным** усреднением (constant step-size: `Q += alpha * (r - Q)`
вместо `Q += (r - Q) / N`). Какой вариант лучше отслеживает изменения среды и почему?


In [ ]:
# TODO (бонус): ваш код здесь


## Часть 2. Теория (30 баллов)

Отвечайте прямо в markdown-ячейках, используя LaTeX ($...$ или $$...$$). Показывайте
промежуточные шаги, а не только финальный ответ — за них тоже начисляются баллы.

### 2.1 Return для простого MDP (10 баллов)

Дан эпизод (последовательность наград после каждого шага):

$$
R_1 = 1,\; R_2 = 0,\; R_3 = 2,\; R_4 = 0,\; R_5 = 1 \quad \text{(эпизод завершается после шага 5)}
$$

Посчитайте $G_0, G_1, \ldots, G_4$ для $\gamma = 0.9$. Покажите, как использовать
рекуррентное соотношение $G_t = R_{t+1} + \gamma G_{t+1}$, чтобы не считать каждую
сумму с нуля.

_Ваш ответ:_ TODO


### 2.2 Вывод уравнения Беллмана для V^π (10 баллов)

Исходя из определения $V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$ и рекуррентного
соотношения $G_t = R_{t+1} + \gamma G_{t+1}$, выведите уравнение Беллмана:

$$
V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V^\pi(s')\right]
$$

Распишите каждый шаг вывода (где именно используется линейность матожидания,
где — марковское свойство, где — определение $\pi$ и $P$).

_Ваш ответ:_ TODO


### 2.3 MDP на бумаге: робот-уборщик (10 баллов)

Робот-уборщик находится в одной из трёх комнат: `{Кухня, Зал, Коридор}`. Действия:
`{Убирать, Переехать}`.

* Из **Кухни**: `Убирать` -> остаться в Кухне с наградой +2 (вероятность 1);
  `Переехать` -> Зал с наградой 0 (вероятность 0.8) или Коридор с наградой 0 (вероятность 0.2)
* Из **Зала**: `Убирать` -> остаться в Зале с наградой +1 (вероятность 1);
  `Переехать` -> Кухня с наградой 0 (вероятность 0.5) или Коридор с наградой 0 (вероятность 0.5)
* Из **Коридора**: `Убирать` -> остаться в Коридоре с наградой 0 (в коридоре убирать нечего);
  `Переехать` -> Кухня с наградой 0 (вероятность 0.5) или Зал с наградой 0 (вероятность 0.5)

Пусть политика π детерминированная: `Убирать` в Кухне и Зале, `Переехать` в Коридоре
(робот не задерживается в коридоре). Возьмите γ = 0.9.

**Задание:** запишите систему из 3 линейных уравнений на $V^\pi(\text{Кухня})$,
$V^\pi(\text{Зал})$, $V^\pi(\text{Коридор})$ по уравнению Беллмана из 2.2, и решите её
(аналитически или через `numpy.linalg.solve` в коде ниже — но составить систему нужно
руками).

_Ваш ответ (система уравнений):_ TODO


In [ ]:
# Опционально: проверьте свой аналитический ответ, решив систему уравнений численно.
# Постройте матрицы так, чтобы задача свелась к V = R_pi + gamma * P_pi @ V,
# то есть (I - gamma * P_pi) @ V = R_pi.

# TODO: задайте P_pi (3x3) и R_pi (3,) согласно политике pi и условию задачи,
# затем решите систему через np.linalg.solve

gamma = 0.9
# P_pi = ...
# R_pi = ...
# V = np.linalg.solve(np.eye(3) - gamma * P_pi, R_pi)
# print(V)


## Чек-лист перед сдачей

- [ ] `DecayingEpsilonGreedyAgent` реализован и проходит self-check
- [ ] Построено сравнение 4 агентов на 3 конфигурациях бандита
- [ ] Отвечен вопрос про разницу между конфигурациями
- [ ] Построен график чувствительности к гиперпараметрам
- [ ] Посчитан return $G_0, \ldots, G_4$
- [ ] Выведено уравнение Беллмана для $V^\pi$ по шагам
- [ ] Составлена и решена система уравнений для робота-уборщика
- [ ] (опционально) реализован non-stationary бандит и сравнение способов усреднения
